# RTX-KG2 one-hop queries — all-Soufflé

## Setup

In [ ]:
import os
import json
import subprocess
import pandas as pd
from dotenv import load_dotenv
import anthropic

pd.set_option("display.max_colwidth", 80)

NODES_TSV = "rtx-kg2_nodes_2.8.1.tsv"
EDGES_TSV = "rtx-kg2_edges_2.8.1.tsv"
PREDICATES_TSV = "distinct_core_predicate.tsv"
CATEGORIES_TSV = "distinct_category.tsv"
SOUFFLE = "souffle"
OUT_DIR = "souffle_out"

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ["APIKEY"])
MODEL = "claude-sonnet-4-5"

def llm(system, user, max_tokens=1024):
    msg = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                 system=system, messages=[{"role": "user", "content": user}])
    return msg.content[0].text

def _json(text):
    i, j = text.find("{"), text.rfind("}")
    return json.loads(text[i:j + 1])

def _souffle_code(text):
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1].rsplit("```", 1)[0]
    return t.strip()

def run_souffle(dl_path, out_dir=OUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    subprocess.run([SOUFFLE, dl_path, "-F", ".", "-D", out_dir], check=True)

def _read_csv(path, names):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return pd.DataFrame(columns=names)
    return pd.read_csv(path, sep="\t", header=None, names=names, dtype=str).fillna("")

## Vocabulary

In [ ]:
def load_predicates():
    df = pd.read_csv(PREDICATES_TSV, sep="\t")
    return df.sort_values("n", ascending=False)["predicate"].astype(str).tolist()

def load_categories():
    if not os.path.exists(CATEGORIES_TSV):
        with open("categories.dl", "w") as f:
            f.write(
                ".decl node(c:symbol)\n"
                '.input node(IO=file, filename="' + NODES_TSV + '", columns="15")\n'
                ".decl categories(c:symbol)\n"
                "categories(c) :- node(c).\n"
                ".output categories\n"
            )
        run_souffle("categories.dl")
        c = _read_csv(os.path.join(OUT_DIR, "categories.csv"), ["category"])
        c = c[c["category"] != ""].drop_duplicates().sort_values("category")
        c.to_csv(CATEGORIES_TSV, sep="\t", header=False, index=False)
    return _read_csv(CATEGORIES_TSV, ["category"])["category"].tolist()

KG2_PREDICATES = load_predicates()
VALID_CATEGORIES = load_categories()
print(str(len(KG2_PREDICATES)) + " predicates | " + str(len(VALID_CATEGORIES)) + " categories")

## Parse the question

In [ ]:
PARSE_SYSTEM = (
    "You convert a natural-language biomedical question into a ONE-HOP knowledge-graph query "
    "over edges of the form (subject)-[predicate]->(object).\n\n"
    "Return ONLY JSON:\n"
    '{\n'
    '  "subject":   {"name": <entity the user named, or null>, "categories": [<biolink category>, ...]},\n'
    '  "predicate": <one biolink predicate, or null>,\n'
    '  "object":    {"name": <entity the user named, or null>, "categories": [<biolink category>, ...]}\n'
    '}\n\n'
    "Rules:\n"
    "- If the user names a specific thing for a slot, put it in \"name\" and give its likely "
    "biolink \"categories\".\n"
    "- If a slot is what the user is ASKING FOR (the unknown), or they say \"any\": \"name\"=null "
    "and \"categories\"=the expected type(s) (or [] if truly anything).\n"
    "- \"predicate\": pick the single best match from this list, or null if the relation is "
    "unconstrained: " + "{predicates}" + ".\n"
    "- Respect direction: biolink:treats goes drug(subject) -> disease(object).\n"
    "- Use ONLY these biolink categories: " + "{categories}" + "."
)

def parse_query(question):
    system = (PARSE_SYSTEM
              .replace("{predicates}", ", ".join(KG2_PREDICATES))
              .replace("{categories}", ", ".join(VALID_CATEGORIES)))
    return _json(llm(system, question, max_tokens=400))

## Resolve entities

In [ ]:
def _ci_regex(term):
    parts = []
    for ch in term.strip():
        if ch.isalpha():
            parts.append("[" + ch.lower() + ch.upper() + "]")
        elif ch in '"]^\\':
            parts.append(".")
        else:
            parts.append("[" + ch + "]")
    return ".*" + "".join(parts) + ".*"

def _rank(name, src, term_lc):
    nl = name.lower()
    if nl == term_lc:
        return 0
    if nl.startswith(term_lc):
        return 1
    if term_lc in nl:
        return 2
    return 3

def resolve_entity(term, categories=None, limit=25):
    pat = _ci_regex(term)
    lines = [
        ".decl node(id:symbol, name:symbol, cat:symbol, syn:symbol)",
        '.input node(IO=file, filename="' + NODES_TSV + '", columns="7:10:15:13")',
        ".decl cand(id:symbol, name:symbol, cat:symbol, src:symbol)",
    ]
    guard = ""
    if categories:
        lines.append(".decl cat_set(c:symbol)")
        for c in categories:
            lines.append('cat_set("' + c + '").')
        guard = ", cat_set(c)"
    lines.append('cand(i, n, c, "name") :- node(i, n, c, _)' + guard + ', match("' + pat + '", n).')
    lines.append('cand(i, n, c, "syn") :- node(i, n, c, s)' + guard + ', match("' + pat + '", s).')
    lines.append(".output cand")
    with open("resolve.dl", "w") as f:
        f.write("\n".join(lines) + "\n")
    run_souffle("resolve.dl")
    df = _read_csv(os.path.join(OUT_DIR, "cand.csv"), ["id", "name", "category", "src"])
    if len(df) == 0:
        return df[["id", "name", "category"]]
    tl = term.strip().lower()
    df["rank"] = [_rank(n, s, tl) for n, s in zip(df["name"], df["src"])]
    df["nlen"] = df["name"].str.len()
    df = df.sort_values(["rank", "nlen"]).drop_duplicates("id").head(limit)
    return df[["id", "name", "category"]].reset_index(drop=True)

SELECT_SYSTEM = (
    "You are given candidate knowledge-graph entities (each a real CURIE) that lexically match a term. "
    "Choose the row number(s) that refer to the entity the user means. If several rows are the same "
    "real-world entity under different vocabularies, choose them all.\n"
    'Return ONLY JSON: {"indices": [<row numbers>]} or {"indices": "all"}.'
)

def llm_select(df, term, role):
    if len(df) == 0:
        return []
    rows = "\n".join("[" + str(i) + "] " + r["name"] + " | " + r["category"] + " | " + r["id"]
                     for i, r in df.iterrows())
    out = _json(llm(SELECT_SYSTEM, "Term: " + term + " (role: " + role + ")\nCandidates:\n" + rows, max_tokens=300))
    idx = out.get("indices")
    if idx == "all":
        chosen = df
    else:
        keep = [int(i) for i in idx if int(i) in df.index]
        chosen = df.loc[keep]
    return chosen.to_dict("records")

## Write the query

In [ ]:
EDGE_HEADER = (
    ".decl edge(subj:symbol, pred:symbol, obj:symbol, neg:symbol)\n"
    f'.input edge(IO=file, filename="{EDGES_TSV}", columns="16:0:17:3")'
)

DL_SYSTEM = (
    "You write Soufflé Datalog for ONE-HOP queries over the RTX-KG2 graph.\n"
    "Every program MUST begin with exactly these two lines:\n"
    + EDGE_HEADER + "\n\n"
    "edge(subj, pred, obj, neg) is one KG edge: subj-[pred]->obj are CURIEs; use only neg=\"False\".\n\n"
    "You are given three slots (subject, predicate, object). Each is FREE or BOUND:\n"
    "- A BOUND ENTITY slot comes as a SET of equivalent CURIEs. Declare a membership relation and "
    "bind that slot to a variable ranging over the set, e.g. for subject:\n"
    "      .decl subject_set(x:symbol)\n"
    "      subject_set(\"ID1\"). subject_set(\"ID2\").\n"
    "  then use subject_set(S) in the rule body with S as the subject. Use the EXACT CURIEs given.\n"
    "- A BOUND PREDICATE is just its exact quoted value.\n"
    "- A FREE slot is a capitalized variable.\n\n"
    "Write ONE rule whose head result(...) holds exactly the FREE slot variables, in "
    "subject,predicate,object order. If NO slot is free, use result(\"yes\").\n"
    "Output ONLY the program (no prose/markdown). End with .output result. "
    'lowercase relation names; avoid the built-in "cat".'
)

def _slot_line(role, plan, bindings):
    if role in bindings:
        ids = bindings[role]["ids"]
        idset = ", ".join(f'"{i}"' for i in ids)
        return f'{role}: BOUND to set {{{idset}}}   ({bindings[role]["name"]}, {len(ids)} id(s))'
    cats = plan[role].get("categories") or []
    return f"{role}: FREE variable" + (f"   (type: {', '.join(cats)})" if cats else "")

def generate_dl_llm(question, plan, bindings, out_path="query.dl"):
    pred = plan.get("predicate")
    spec = "\n".join([
        _slot_line("subject", plan, bindings),
        f'predicate: "{pred}"' if pred else "predicate: FREE variable",
        _slot_line("object", plan, bindings),
    ])
    user = f"Question: {question}\n\nOne-hop slots:\n{spec}\n\nWrite the Soufflé program."
    dl = _souffle_code(llm(DL_SYSTEM, user, max_tokens=800))
    with open(out_path, "w") as f:
        f.write(dl + "\n")
    print(f"Wrote {out_path}\n\n{dl}")
    return out_path

## Run and name

In [ ]:
def name_results(ids):
    if not ids:
        return pd.DataFrame(columns=["id", "name", "category"])
    lines = [
        ".decl node(id:symbol, name:symbol, cat:symbol)",
        '.input node(IO=file, filename="' + NODES_TSV + '", columns="7:10:15")',
        ".decl want(id:symbol)",
    ]
    for i in ids:
        lines.append('want("' + i + '").')
    lines += [
        ".decl named(id:symbol, name:symbol, cat:symbol)",
        "named(i, n, c) :- node(i, n, c), want(i).",
        ".output named",
    ]
    with open("name.dl", "w") as f:
        f.write("\n".join(lines) + "\n")
    run_souffle("name.dl")
    return _read_csv(os.path.join(OUT_DIR, "named.csv"), ["id", "name", "category"])

def run_query(dl_path="query.dl"):
    run_souffle(dl_path)
    path = os.path.join(OUT_DIR, "result.csv")
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        print("No results.")
        return pd.DataFrame()
    rows = pd.read_csv(path, sep="\t", header=None, dtype=str)
    ids = rows[0].astype(str).tolist()
    named = name_results(ids)
    print(str(len(named)) + " result(s):")
    return named

## Pipeline

In [ ]:
query = input("Ask a one-hop question (e.g. 'what drugs treat parkinson disease?'): ").strip()

plan = parse_query(query)
print("Parsed one-hop plan:\n", json.dumps(plan, indent=2), "\n")

bindings = {}
for role in ["subject", "object"]:
    name = plan.get(role, {}).get("name")
    if name:
        cands = resolve_entity(name, plan[role].get("categories"))
        display(cands)
        picks = llm_select(cands, name, role)
        bindings[role] = {"name": name, "ids": [r["id"] for r in picks]}
        print("  " + role + ": " + str(len(bindings[role]["ids"])) + " id(s) selected for '" + name + "'\n")

generate_dl_llm(query, plan, bindings)
answer = run_query("query.dl")
display(answer)